# Adversarial Signal Decomposition and Selective Debiasing Pilot

## 1. Research Question

**Fixed case:** context C = "baseball bat", target T = "sports ball", model = `llava-hf/llava-1.5-7b-hf`.

- **Question A (signal decomposition feasibility):** does the representation shift induced by a targeted
  adversarial attack contain a separable component that selectively supports the spurious
  baseball-bat -> sports-ball hallucination effect?
- **Question B (method feasibility):** if such a component exists, can we train the model to suppress
  it and obtain *better selective* functional debiasing than simple clean/adversarial fine-tuning?

This notebook only loads already-saved outputs -- it does not rerun PGD, hidden-state extraction, LoRA
training, or model evaluation.

## 2. Prior Negative Result

Stage 11 Exp6 (`CooccurrenceHallucinationDiagnostic/scripts/run_stage11_exp6_causal_intervention.py`)
subtracted a single fixed Layer-19 direction (diff-in-means, original vs. bat-removed) from every
position's hidden state. Result: reduction in unsupported Ball evidence (0.257) was **essentially the
same size** as the reduction in genuine Ball evidence (0.275/0.256) and Bat recognition (0.320, actually
larger) -- classified **non-selective**. This pilot asks whether decomposing the shift into multiple
components (rather than assuming one mean direction) can recover a more selective candidate.

## 3. Pilot Dataset

Reused verbatim from `adversarial_functional_debiasing_pilot` (same seed=42 split, same 4 disjoint COCO groups). See `audit/repository_audit.md`.

In [ ]:
import csv, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

OUT = Path("/data3/KJE/code/UQ/outputs/adversarial_signal_debiasing_pilot")

manifest = pd.read_csv(OUT / "data" / "sample_manifest.csv")
manifest.groupby(["fold", "role"])["image_id"].nunique()


## 4. Adversarial Exposure

Existing PGD attack (epsilon=16/255), reused unchanged.

In [ ]:
import csv, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

OUT = Path("/data3/KJE/code/UQ/outputs/adversarial_signal_debiasing_pilot")

adv = pd.read_csv(OUT / "data" / "adversarial_forget_set.csv")
print(f"N={len(adv)}  attack_success_rate={adv.attack_success.mean():.3f}")
print(f"mean clean_s_ball={adv.clean_s_ball.mean():.4f}  mean adv_s_ball={adv.adv_s_ball.mean():.4f}  mean delta_s_ball={adv.delta_s_ball.mean():.4f}")
display(Image(filename=str(OUT / "figures" / "fig1_adversarial_exposure.png")))


## 5. Layer-19 Representation Shift

`delta_h = h_adv - h_clean` at Layer 19 (last teacher-forced decision-token position), for every TRAIN G10 image.

In [ ]:
import csv, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

OUT = Path("/data3/KJE/code/UQ/outputs/adversarial_signal_debiasing_pilot")

import torch
cached = torch.load(OUT / "cached_activations" / "layer19_delta_h.pt")
meta = pd.read_csv(OUT / "cached_activations" / "layer19_metadata.csv")
print(f"delta_h shape: {tuple(cached['delta_h'].shape)}  layer_idx={cached['layer_idx']}")
print(f"mean ||delta_h||: {cached['delta_h'].norm(dim=1).mean().item():.4f}")
meta.describe()


## 6. PCA/SVD Decomposition

Fit on the DEV portion (70%) of TRAIN G10 delta_h only. PC1 is not assumed spurious merely because it explains the most variance -- see Section 7.

In [ ]:
import csv, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

OUT = Path("/data3/KJE/code/UQ/outputs/adversarial_signal_debiasing_pilot")

ev = pd.read_csv(OUT / "decomposition" / "explained_variance.csv")
display(ev)
display(Image(filename=str(OUT / "figures" / "fig2_explained_variance.png")))


## 7. Component Functional Selectivity

Projection intervention `h' = h - lambda*proj_u(h)` at Layer 19, lambda in {0.5, 1.0}, evaluated on the internal VAL split (never on the final CLEAN TEST split).

In [ ]:
import csv, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

OUT = Path("/data3/KJE/code/UQ/outputs/adversarial_signal_debiasing_pilot")

interv = pd.read_csv(OUT / "component_intervention" / "intervention_results.csv")
display(interv)
display(Image(filename=str(OUT / "figures" / "fig3_component_selectivity.png")))


## 8. Mean Direction and Random-Direction Baselines

In [ ]:
import csv, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

OUT = Path("/data3/KJE/code/UQ/outputs/adversarial_signal_debiasing_pilot")

mean_dir = pd.read_csv(OUT / "component_intervention" / "mean_direction_results.csv")
random_dir = pd.read_csv(OUT / "component_intervention" / "random_direction_results.csv")
print("Mean direction:"); display(mean_dir)
print(f"Random directions (n={random_dir.candidate_id.nunique()}): selectivity range "
      f"[{random_dir.selectivity.min():.4f}, {random_dir.selectivity.max():.4f}]")


## 9. Best Component

Ranked by `Selectivity_k = |Delta_spurious| - |Delta_target| - |Delta_context|`, minimum over the two lambda values (must be selective at both), on the internal VAL split -- selection never touches the final CLEAN TEST split.

In [ ]:
import csv, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

OUT = Path("/data3/KJE/code/UQ/outputs/adversarial_signal_debiasing_pilot")

sel = pd.read_csv(OUT / "decomposition" / "component_selectivity.csv")
display(sel)
display(Image(filename=str(OUT / "figures" / "fig4_mean_vs_best_component.png")))


## 10. Clean Debias vs. Adv Debias

Models A and B, reused verbatim from `adversarial_functional_debiasing_pilot` (identical checkpoints, identical clean-test evaluation).

In [ ]:
import csv, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

OUT = Path("/data3/KJE/code/UQ/outputs/adversarial_signal_debiasing_pilot")

summary = pd.read_csv(OUT / "evaluation" / "summary.csv")
display(summary[summary.method.isin(["original", "clean_debias", "adv_debias"])])


## 11. Adv + Decomp Debias

Model C -- the only newly trained LoRA adapter in this pilot.

In [ ]:
import csv, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

OUT = Path("/data3/KJE/code/UQ/outputs/adversarial_signal_debiasing_pilot")

train_cfg = json.loads((OUT / "adv_decomp_debias" / "training_config.json").read_text())
print(json.dumps(train_cfg, indent=2))
train_log = pd.read_csv(OUT / "adv_decomp_debias" / "train_log.csv")
train_log.tail()


## 12. Functional Coupling B

`B = E[s_ball | Bat+, Ball-] - E[s_ball | Bat-, Ball-]`, all four models, unseen CLEAN test images.

In [ ]:
import csv, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

OUT = Path("/data3/KJE/code/UQ/outputs/adversarial_signal_debiasing_pilot")

display(summary[["method", "coupling_B", "delta_coupling_B_from_original"]])
display(Image(filename=str(OUT / "figures" / "fig5_coupling_by_method.png")))


## 13. Genuine Ball Retention (GT, Ball+)

In [ ]:
import csv, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

OUT = Path("/data3/KJE/code/UQ/outputs/adversarial_signal_debiasing_pilot")

display(summary[["method", "ball_plus_acc", "delta_ball_plus_acc_from_original"]])


## 14. Bat Retention (GC, Bat+)

In [ ]:
import csv, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

OUT = Path("/data3/KJE/code/UQ/outputs/adversarial_signal_debiasing_pilot")

display(summary[["method", "bat_plus_acc", "delta_bat_plus_acc_from_original"]])
display(Image(filename=str(OUT / "figures" / "fig6_method_selectivity.png")))


## 15. GO / NO-GO Decision

In [ ]:
import csv, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

OUT = Path("/data3/KJE/code/UQ/outputs/adversarial_signal_debiasing_pilot")

stats = json.loads((OUT / "evaluation" / "statistics.json").read_text())
print("Component selectivity summary:", json.dumps(stats["component_selectivity_summary"], indent=2))
print("\nDecision:", stats["go_no_go"]["decision"])


## 16. Limitations

- Single pair (baseball bat -> sports ball), single model (LLaVA-1.5-7B), single layer (19) -- a feasibility
  pilot, not a general result.
- Component selection used an internal 30% VAL split carved from TRAIN (n=14 per group) -- small, so
  selectivity estimates are noisy; see bootstrap CIs where reported.
- The `Selectivity_k` ranking metric can favor a direction that changes *nothing* (near-zero effect on all
  three groups) over one that changes something with partial selectivity, since both minimize
  `|target|+|context|` but the former also minimizes `|spurious|` trivially. Random directions with
  near-zero effect can therefore rank above real components -- read the raw `Delta_spurious` /
  `Delta_target` / `Delta_context` columns, not just the ranking, before concluding a component is useful.
- PLS was run only because PCA showed weak/poor selectivity (Part VIII's conditional trigger), per the
  pilot's pre-specified decision rule, not as post-hoc tuning.
- No causal claim beyond correlation-plus-intervention at one layer, one token position.

## 17. Next Step

See `README.md`'s "Next Step" section for the concrete recommendation conditioned on the GO/NO-GO decision
reached above.